# Huang Lab — Tissue-on-a-Chip MAE + Detection Training

**Before running:**
- Runtime → Change runtime type → **T4 GPU**
- Make sure `250918_Deepmind_CV_Collaboration` is added as a shortcut in your My Drive
- Have your W&B API key ready from https://wandb.ai/authorize
- Have a GitHub Personal Access Token ready (Settings → Developer settings → Personal access tokens → Tokens classic → New token, check **repo** scope)

## 1. Check GPU

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU — go to Runtime → Change runtime type → T4 GPU')

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_ROOT = '/content/drive/MyDrive/250918_Deepmind_CV_Collaboration'
assert os.path.isdir(DATA_ROOT), (
    f'Data folder not found at {DATA_ROOT}.\n'
    'Go to drive.google.com → Shared with me → right-click '
    '250918_Deepmind_CV_Collaboration → Organize → Add shortcut → My Drive'
)
print('✓ Data root found:', DATA_ROOT)
print(os.listdir(DATA_ROOT))

## 3. Clone repo (private — needs GitHub token)

Get a token at: https://github.com/settings/tokens/new  
Check the **repo** scope, generate, and paste below.

In [ ]:
from google.colab import userdata
# Securely prompt for GitHub token (not stored in notebook)
import getpass
GITHUB_TOKEN = getpass.getpass('Paste your GitHub Personal Access Token: ')

import subprocess, sys
result = subprocess.run(
    ['git', 'clone', f'https://{GITHUB_TOKEN}@github.com/racyun/Huang-Lab-Work.git',
     '/content/Huang-Lab-Work'],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('ERROR:', result.stderr)
else:
    print('✓ Repo cloned successfully')
    
%cd /content/Huang-Lab-Work

## 4. Install dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q transformers accelerate timm
print('✓ Dependencies installed')

## 5. W&B login

In [ ]:
import wandb, getpass
WANDB_KEY = getpass.getpass('Paste your W&B API key (from https://wandb.ai/authorize): ')
wandb.login(key=WANDB_KEY)
print('✓ W&B logged in')

## 6. Write Colab config

In [ ]:
import os
DATA_ROOT = '/content/drive/MyDrive/250918_Deepmind_CV_Collaboration'

colab_config = f"""# Auto-generated Colab config — do not commit.

dataset:
  well_prefix: "W"
  well_count: 222
  zstack_subdir: "P00001"
  expected_z_slices: 140
  hybrid_folder_template: "hybrid_results_{{well_id}}"
  focus_filename_glob: "*focus_stacked.tif"
  cache_dir: "/content/huang_lab_cache"
  resize:
    zstack:  [64, 64]
    focused: [224, 224]
    hybrid:  [224, 224]

  splits:
    - name: "900kpa"
      stiffness_kpa: 900.0
      zstack_root:  "{DATA_ROOT}/250811_Athchip_noninflam_900kPa"
      focused_root: "{DATA_ROOT}/20251027_2123__FocusStack_250811_Athchip_noninflam_900kPa"
      hybrid_root:  "{DATA_ROOT}/250811_Athchip_noninflam_900kPa_HybridResults"
      labels_root:  "{DATA_ROOT}/bbox_txt_for_training/900kPa"

    - name: "5kpa"
      stiffness_kpa: 5.0
      zstack_root:  "{DATA_ROOT}/250814_Athchip_non-inflam_5kPa"
      focused_root: "{DATA_ROOT}/20251027_2215__FocusStack_250814_Athchip_non-inflam_5kPa"
      hybrid_root:  "{DATA_ROOT}/250814_Athchip_non-inflam_5kPa_HybridResults"
      labels_root:  "{DATA_ROOT}/bbox_txt_for_training/5kPa"

training:
  output_dir: "/content/outputs"
  batch_size: 4
  num_workers: 2
  epochs: 50
  lr: 1.5e-4
  warmup_epochs: 5
  amp: true
  grad_clip: 1.0

detection:
  epochs: 50
  batch_size: 4
  num_workers: 2
  amp: true
  train_image_short_side: 800
  conf_threshold: 0.3

wandb:
  enabled: true
  project: "huang-lab-tissue-chip"
  entity: null
  log_freq: 5
"""

with open('config/colab.yaml', 'w') as f:
    f.write(colab_config)
print('✓ config/colab.yaml written')

## 7. Verify one batch loads

In [ ]:
!python3 scripts/train_pretrain.py \
    --config config/default.yaml \
    --local-config config/colab.yaml \
    --inspect-data

## 8. Pretrain — Multi-encoder MAE (50 epochs)

Trains three encoders jointly with stiffness conditioning.  
W&B logs: `train/loss`, `train/loss_focused`, `train/loss_hybrid`, `train/loss_volume`, per-encoder LRs.

In [ ]:
!python3 scripts/train_pretrain.py \
    --config config/default.yaml \
    --local-config config/colab.yaml \
    --train \
    --wandb \
    --wandb-run-name "full-222well-pretrain-50ep"

## 9. Detection fine-tuning — Deformable-DETR (50 epochs)

W&B logs: `detect/loss`, `eval/mAP`, `eval/AP50`, `eval/AP75`, `eval/mean_iou`.

In [ ]:
import glob
ckpts = sorted(glob.glob('/content/outputs/pretrain/*.pth'))
print('Latest pretrain checkpoint:', ckpts[-1] if ckpts else 'none (will use COCO weights)')

In [ ]:
!python3 scripts/train_detect.py \
    --config config/default.yaml \
    --local-config config/colab.yaml \
    --wandb \
    --wandb-run-name "full-222well-detect-50ep"

## 10. Save outputs to Drive

In [ ]:
import shutil, os
SAVE_DIR = f'{DATA_ROOT}/colab_outputs'
os.makedirs(SAVE_DIR, exist_ok=True)
shutil.copytree('/content/outputs', SAVE_DIR, dirs_exist_ok=True)
print('✓ Outputs saved to:', SAVE_DIR)

## Tips

**If Colab disconnects mid-run:** Resume with `--resume /content/outputs/pretrain/checkpoint_epochN.pth`

**Expected T4 training times:**
- Pretrain: ~8–15 min/epoch × 50 epochs ≈ 7–12 hours total
- Detection: ~5–10 min/epoch × 50 epochs ≈ 4–8 hours total
- Use **Colab Pro** for uninterrupted long sessions

**Expected metrics after 50 epochs on full dataset:**
- MAE loss < 0.01
- Detection AP50: 0.3–0.6
- Detection mAP@[0.5:0.95]: 0.15–0.35

**W&B dashboard:** https://wandb.ai/models-fusionai/huang-lab-tissue-chip